# 🚀 FaceKey Animation Studio — Kaggle Data Collector & GPU Trainer
### Parallel Multi-Cloud Execution (Concurrent with Google Colab)

**Why Kaggle?**
- **30 Hours of Free GPU Weekly** (NVIDIA P100 16GB or 2x NVIDIA T4 16GB).
- **Turbo Multi-Worker Processing**: 4 CPU cores allow running `--num-workers 2` or 3 concurrently at 150+ FPS!
- **Runs Simultaneously with Google Colab**: Colab and Kaggle process different batches in parallel, multiplying your data collection throughput by **20x–25x**!
- **Distributed Locking**: No redundant downloads! Atomic locks in Google Drive ensure Colab and Kaggle never process the same video.
- **Session Expiry Recovery**: If a Kaggle 12-hour session ends or is preempted, locks are automatically released or reclaimed on next start.

# ⚡ 1-CLICK INSTANT RUNNER (RUN THIS SINGLE CELL ON KAGGLE!)
> [!TIP]
> **Whenever you start Kaggle, simply click ▶️ Play on THIS SINGLE CELL below (or press `Ctrl+Shift+Enter`)!**
> It updates code from GitHub, sets up storage, installs dependencies, and launches 3 workers on Kaggle's 4 CPU cores in parallel with Colab!

In [ ]:
# ==============================================================================
# 🚀 FACEKEY ALL-IN-ONE 1-CLICK RUNNER FOR KAGGLE
# ==============================================================================
import os, sys, shutil

# 1. Detect / Configure Dataset Root
possible_paths = [
    "/kaggle/input/google-drive/FaceKeyDataset",
    "/kaggle/working/FaceKeyDataset"
]
DRIVE_DIR = "/kaggle/working/FaceKeyDataset"
for p in possible_paths:
    if os.path.exists(p):
        DRIVE_DIR = p
        break

for folder in ("chunks", "checkpoints", "locks", "data"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)
print(f"[+] Using Dataset Root: {DRIVE_DIR}")

# 2. Update to latest repository code (Self-Healing)
%cd /kaggle/working
APP_DIR = "/kaggle/working/FaceKeyAnimation"
if os.path.isdir(os.path.join(APP_DIR, ".git")):
    print("[*] Updating FaceKeyAnimation to latest GitHub main...")
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print("[*] Fresh clone of FaceKeyAnimation from GitHub...")
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Fast Cloud Dependencies & Deno JS solver
!pip install -q -U --pre "yt-dlp[default]" mediapipe opencv-python-headless numpy sounddevice mss av scipy
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Sync Kaggle Queue
urls_file = os.path.join(APP_DIR, "sessions", "youtube_urls_kaggle.txt")

# 5. Launch Collector (3 Workers across Kaggle 4 CPU cores, 1 Face Only, Turbo Mode)
print("\n[*] Launching FaceKey Turbo Multi-Worker Kaggle Collector...")
!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "kaggle-worker-1" \
    --urls-file "{urls_file}" \
    --chunk-size 5 \
    --quality 480p \
    --num-workers 3


---
# 📦 KAGGLE STORAGE MANAGEMENT & CLOUD-TO-CLOUD BACKUP (0 MB LOCAL INTERNET)
> [!TIP]
> **Kaggle Disk Strategy (< 20 GB Quota)**:
> Each 5-session chunk is only ~20 MB. You can easily collect 100+ chunks before reaching even 5 GB!
> Whenever you want to transfer chunks to your 5 TB Google Drive without downloading to your local PC, run the upload cells below.
> Once uploaded, local chunks are automatically purged from Kaggle so your disk stays clean.

In [ ]:
# ==============================================================================
# 📊 CHECK KAGGLE DISK USAGE & CHUNKS SUMMARY
# ==============================================================================
%cd /kaggle/working/FaceKeyAnimation
!python scripts/cloud_drive_uploader.py --method status


In [ ]:
# ==============================================================================
# ☁️ METHOD 1: DIRECT UPLOAD TO 5 TB GOOGLE DRIVE (SERVICE ACCOUNT)
# ==============================================================================
# 1. Create a free Google Cloud Service Account & download JSON key.
# 2. Share your 'FaceKeyDataset' folder on Google Drive with the service account email as 'Editor'.
# 3. Set the JSON path and your Folder ID below and click Play.
# Transferred cloud-to-cloud: 0 MB home data used!

SERVICE_ACCOUNT_JSON = '/kaggle/working/service_account.json'
GOOGLE_DRIVE_FOLDER_ID = 'YOUR_GOOGLE_DRIVE_FOLDER_ID'

!python scripts/cloud_drive_uploader.py \
    --method gdrive \
    --service-account "{SERVICE_ACCOUNT_JSON}" \
    --folder-id "{GOOGLE_DRIVE_FOLDER_ID}"


In [ ]:
# ==============================================================================
# 🚀 METHOD 2: ZERO-GCP HUGGINGFACE HUB BRIDGE (FREE UNLIMITED PRIVATE STORAGE)
# ==============================================================================
# 1. Get a free write token from https://huggingface.co/settings/tokens
# 2. Put your token and dataset name below and click Play.
# Chunks upload to HuggingFace directly from Kaggle.
# Then in Colab, run the 1-line pull cell to move them into 5 TB Google Drive!

HF_TOKEN = 'YOUR_HF_WRITE_TOKEN'
HF_REPO = 'YOUR_USERNAME/facekey-chunks'

!python scripts/cloud_drive_uploader.py \
    --method hf \
    --hf-token "{HF_TOKEN}" \
    --hf-repo "{HF_REPO}"


---
# ⚠️ IMPORTANT NOTES ON KAGGLE EXECUTION & GOOGLE DRIVE PERSISTENCE:
> [!NOTE]
> **Why Subsequent Cells Are Waiting**:
> When Cell 1 is running, it continuously downloads, extracts, and packages videos. In Jupyter/Kaggle, cells execute sequentially, so Steps 1–4 remain in the 'waiting' queue while Cell 1 is active. You do **NOT** need to run Steps 1–3 if you ran Cell 1!
>
> **Google Drive Storage & Zero Home Internet**:
> Kaggle does not support native write mounting to Google Drive like Colab does. Files are stored locally in `/kaggle/working/FaceKeyDataset/chunks/`.
> **Recommended Strategy**: Use **Dual Colab Accounts** (2 Google accounts running simultaneously) for 100% native, real-time writing directly to your 5 TB Google Drive with 0 MB home data.

## Step 1: Verify Kaggle GPU Accelerator & Install Dependencies (Self-Healing)
In the Kaggle Notebook sidebar: Ensure **Accelerator** is set to **GPU P100** or **GPU T4 x2**, and **Internet** is set to **Internet On**.

In [ ]:
!nvidia-smi

import os, sys, shutil

# Always reset working directory to root /kaggle/working first
%cd /kaggle/working

REPO_URL = 'https://github.com/Vijay-1010110/FaceKeyAnimation.git'
APP_DIR = '/kaggle/working/FaceKeyAnimation'

# Self-healing repository clone/reset
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating existing repository to latest origin/main...')
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print('[*] Performing fresh clone of FaceKeyAnimation...')
    !rm -rf {APP_DIR}
    !git clone {REPO_URL} {APP_DIR}

%cd {APP_DIR}

# Install dependencies
!pip install -q -U --pre "yt-dlp[default]" mediapipe opencv-python-headless numpy sounddevice mss av scipy torch torchvision torchaudio
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi
print('[+] Kaggle environment and repository ready!')

## Step 2: Configure Google Drive / Cloud Storage on Kaggle
Kaggle allows connecting Google Drive through multiple methods:
1. **Add Storage → Google Drive**: In the right sidebar, click **+ Add Input**, select **Google Drive**, and authenticate. Drive mounts under `/kaggle/input/google-drive`.
2. **Custom / Working Directory**: If running standalone on Kaggle, data will be organized in `/kaggle/working/FaceKeyDataset`.

In [ ]:
# Detect Drive location on Kaggle
possible_drive_paths = [
    '/kaggle/input/google-drive/FaceKeyDataset',
    '/kaggle/input/facekeydataset/FaceKeyDataset',
    '/kaggle/working/google_drive/FaceKeyDataset',
    '/kaggle/working/FaceKeyDataset'
]

DRIVE_DIR = '/kaggle/working/FaceKeyDataset'
for p in possible_drive_paths:
    if os.path.exists(p):
        DRIVE_DIR = p
        break

os.makedirs(f'{DRIVE_DIR}/chunks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/locks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)

print(f'[+] Using Dataset Root: {DRIVE_DIR}')

## Step 3: Run Turbo Multi-Worker Batch Collector on Kaggle
Kaggle processes its dedicated queue `youtube_urls_kaggle.txt` (Categories: MIT OCW, Stanford Online, Closer to Truth, Lex Fridman, Numberphile):
- **Turbo Scratch Mode**: Downloads in ~2s, decodes at 150–220 FPS.
- **2 Concurrent Workers**: Spawns 2 parallel workers across Kaggle's 4 CPU cores.
- **Operates in lock-step with Colab**: Shared atomic locks prevent any duplicate processing.
- **Session Expiry Recovery**: If session ends after 12 hrs, locks are automatically reclaimed.

In [ ]:
urls_file = os.path.join(APP_DIR, 'sessions', 'youtube_urls_kaggle.txt')

!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "kaggle-worker-1" \
    --urls-file "{urls_file}" \
    --chunk-size 5 \
    --quality 480p \
    --num-workers 3

## Step 4: Run Fast GPU Model Training on Kaggle (P100 / Dual T4)
Unpack chunks to Kaggle's high-speed NVMe scratch disk and train the `SpeechToFaceAnimator` model with Automatic Mixed Precision (FP16).

In [ ]:
# Unpack chunks to fast local NVMe SSD
LOCAL_DATA_DIR = '/kaggle/working/training_data'

from src.storage.cloud_sync import CloudDriveSync
sync = CloudDriveSync(project_root=APP_DIR, drive_folder=DRIVE_DIR)
sync.unpack_all_chunks_to_local_ssd(LOCAL_DATA_DIR)

# Launch PyTorch Model Training
!python scripts/train_speech_to_animation.py \
    --drive-dir "{DRIVE_DIR}" \
    --batch-size 64 \
    --epochs 50 \
    --seq-len 64